# Notebook 3: TextCNN

**Research question:** Can a spam classifier trained on SMS messages generalize to emails?

This notebook introduces a neural text classifier trained from scratch. It uses the same data splits and evaluation protocol as the TF-IDF baseline, allowing a direct comparison of model architecture.

## Experiment design

- Learn the vocabulary from the training split only.
- Truncate or pad messages to 256 tokens and cap the vocabulary at 30,000 tokens.
- Use trainable embeddings followed by a one-dimensional convolution and global max pooling.
- Apply balanced class weights and early stopping on validation loss.
- Train one model on SMS and another on Enron, then evaluate all four domain pairs.
- Keep the configuration fixed; the test sets are not used for tuning.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/kbozukov-coke/cross-domain-spam-detection.git'
PROJECT_NAME = 'cross-domain-spam-detection'
IS_KAGGLE = Path('/kaggle/working').exists()

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working') / PROJECT_NAME
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)],
            check=True,
        )
    else:
        subprocess.run(
            ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'],
            check=True,
        )
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay

from src.data import load_prepared_splits, summarize_splits
from src.modeling import run_transfer_experiments
from src.textcnn import run_textcnn_experiments, textcnn_prediction_details

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)
RANDOM_STATE = 42

print(f'Running in: {"Kaggle" if IS_KAGGLE else "local environment"}')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU devices: {tf.config.list_physical_devices("GPU")}')

## Load the prepared data

The preprocessing and splits are identical to the previous notebooks.

In [ ]:
splits, cleaning_audit = load_prepared_splits(random_state=RANDOM_STATE)
split_summary = summarize_splits(splits)
split_summary.loc[:, ['dataset', 'split', 'rows', 'ham', 'spam', 'spam_rate']]

## Fixed TextCNN configuration

The model is intentionally compact. The convolution captures local token patterns, while global max pooling retains the strongest signal from each filter regardless of message position.

In [ ]:
TEXTCNN_CONFIG = {
    'random_state': RANDOM_STATE,
    'max_tokens': 30_000,
    'sequence_length': 256,
    'embedding_dim': 128,
    'filters': 128,
    'kernel_size': 5,
    'dense_units': 64,
    'dropout': 0.4,
    'learning_rate': 1e-3,
    'epochs': 8,
    'batch_size': 64,
    'patience': 2,
    'verbose': 1,
}

pd.Series(TEXTCNN_CONFIG, name='value').to_frame()

## Train on each domain

Only each source's training split adapts its vocabulary and model weights. Validation data controls early stopping.

In [ ]:
textcnn_models, training_histories, textcnn_results = run_textcnn_experiments(
    splits,
    **TEXTCNN_CONFIG,
)
print('Finished training the SMS and Enron TextCNN models.')

## Learning curves

The training and validation loss curves are checked for convergence and overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for axis, domain in zip(axes, ['sms', 'enron']):
    history = training_histories[domain].history
    epochs = range(1, len(history['loss']) + 1)
    axis.plot(epochs, history['loss'], marker='o', label='train')
    axis.plot(epochs, history['val_loss'], marker='o', label='validation')
    axis.set_title(f'{domain.upper()} training loss')
    axis.set_xlabel('Epoch')
    axis.set_ylabel('Binary cross-entropy')
    axis.legend()

plt.tight_layout()
plt.show()

## TextCNN results

F1 remains the primary metric. Training metadata is included to make the computational cost visible.

In [ ]:
metric_columns = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
display_results = textcnn_results.copy()
display_results[metric_columns] = display_results[metric_columns].round(3)
display_results['training_seconds'] = display_results['training_seconds'].round(1)
display_results.loc[:, [
    'train_domain', 'test_domain', 'setting', 'test_rows',
    *metric_columns, 'epochs_trained', 'training_seconds', 'parameters'
]]

## Compare with the ML baseline

The baseline is refitted on the same training data so the comparison is self-contained.

In [ ]:
baseline_models, baseline_results = run_transfer_experiments(
    splits, random_state=RANDOM_STATE
)
baseline_results = baseline_results.assign(model='TF-IDF + Logistic Regression')

comparison = pd.concat(
    [
        baseline_results.loc[:, ['model', 'train_domain', 'test_domain', *metric_columns]],
        textcnn_results.loc[:, ['model', 'train_domain', 'test_domain', *metric_columns]],
    ],
    ignore_index=True,
)
comparison['experiment'] = (
    comparison['train_domain'].str.upper()
    + ' -> '
    + comparison['test_domain'].str.upper()
)

comparison_table = comparison.pivot(
    index='experiment', columns='model', values='f1'
).reindex(['SMS -> SMS', 'SMS -> ENRON', 'ENRON -> SMS', 'ENRON -> ENRON'])
comparison_table.round(3)

In [ ]:
plt.figure(figsize=(10, 5))
axis = sns.barplot(data=comparison, x='experiment', y='f1', hue='model')
axis.set_title('F1: ML baseline vs TextCNN')
axis.set_xlabel('Train -> test domain')
axis.set_ylabel('F1 score')
axis.set_ylim(0, 1)
axis.legend(title='Model', loc='lower right')
for container in axis.containers:
    axis.bar_label(container, fmt='%.3f', padding=3)
plt.tight_layout()
plt.show()

## TextCNN confusion matrices

The error balance shows whether transfer mainly increases missed spam or false alarms.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for axis, row in zip(axes.ravel(), textcnn_results.itertuples(index=False)):
    test_frame = splits[row.test_domain]['test']
    scores = textcnn_models[row.train_domain].predict(
        test_frame['text'].astype(str).to_numpy(dtype=object), verbose=0
    ).reshape(-1)
    predictions = (scores >= 0.5).astype(np.int8)
    ConfusionMatrixDisplay.from_predictions(
        test_frame['label'],
        predictions,
        display_labels=['ham', 'spam'],
        colorbar=False,
        ax=axis,
    )
    axis.set_title(f'Train {row.train_domain.upper()} -> Test {row.test_domain.upper()}')

plt.tight_layout()
plt.show()

## SMS -> Enron error analysis

High-confidence errors reveal which email patterns are not captured by the SMS-trained network.

In [ ]:
sms_to_enron = textcnn_prediction_details(
    textcnn_models['sms'], splits['enron']['test']
)
cross_domain_errors = sms_to_enron.loc[~sms_to_enron['correct']].copy()
cross_domain_errors['error_type'] = cross_domain_errors['label'].map(
    {0: 'false positive', 1: 'false negative'}
)
cross_domain_errors['confidence'] = (cross_domain_errors['spam_probability'] - 0.5).abs()

display(cross_domain_errors['error_type'].value_counts().rename('errors').to_frame())
cross_domain_errors.sort_values('confidence', ascending=False).loc[
    :, ['error_type', 'spam_probability', 'text']
].head(10)

## TextCNN conclusion

The transfer gap and the change relative to Logistic Regression determine whether learned embeddings and local neural features improve generalization.

In [ ]:
def f1_for(frame, train_domain, test_domain):
    return frame.query(
        'train_domain == @train_domain and test_domain == @test_domain'
    )['f1'].iloc[0]

baseline_sms_to_enron = f1_for(baseline_results, 'sms', 'enron')
textcnn_sms_to_sms = f1_for(textcnn_results, 'sms', 'sms')
textcnn_sms_to_enron = f1_for(textcnn_results, 'sms', 'enron')
textcnn_transfer_gap = textcnn_sms_to_sms - textcnn_sms_to_enron
improvement = textcnn_sms_to_enron - baseline_sms_to_enron

print(f'TextCNN SMS -> SMS F1:       {textcnn_sms_to_sms:.3f}')
print(f'TextCNN SMS -> Enron F1:     {textcnn_sms_to_enron:.3f}')
print(f'TextCNN transfer gap:        {textcnn_transfer_gap:.3f}')
print(f'Change vs ML cross-domain:   {improvement:+.3f}')

if improvement > 0:
    print('Conclusion: TextCNN improves SMS-to-email transfer over the ML baseline.')
else:
    print('Conclusion: TextCNN does not improve SMS-to-email transfer over the ML baseline.')